# Validating Matches with Ground Truth

## Setup and Imports

In [6]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

In [7]:
import os
os.chdir('/Users/graceegeorge/ds6015_know_their_names')

In [8]:
predictions = pd.read_parquet(
    "data/boosted_preds.parquet",
    engine="fastparquet"
)


In [9]:
predictions.head()

,unique_id_l,unique_id_r,match_probability,relatives_match_probability
0,ALB-CN-1870-10570,ALB-CN-1880-6166,0.999702,NaN
1,ALB-CN-1870-12185,ALB-CN-1880-6171,0.999702,NaN
2,ALB-CN-1870-14277,ALB-CN-1880-6173,0.999702,NaN
3,ALB-CN-1870-18074,ALB-CN-1880-6174,0.999702,NaN
4,ALB-CN-1870-14278,ALB-CN-1880-6175,0.999702,NaN


In [10]:
truth = pd.read_csv('data/ground_truth.csv')
truth['1870_id']= 'ALB-CN-1870-' + truth['1870_line'].astype(str)
truth['1880_id']= 'ALB-CN-1880-' + truth['1880_line'].astype(str)

In [11]:
truth.head()

,1870_line,1880_line,score,confidence,1870_id,1880_id
0,1688,22721,495,3,ALB-CN-1870-1688,ALB-CN-1880-22721
1,1695,22737,480,3,ALB-CN-1870-1695,ALB-CN-1880-22737
2,1693,22735,470,3,ALB-CN-1870-1693,ALB-CN-1880-22735
3,1692,22734,460,3,ALB-CN-1870-1692,ALB-CN-1880-22734
4,17144,25766,455,3,ALB-CN-1870-17144,ALB-CN-1880-25766


In [12]:
mentions = pd.read_csv('data/mentions.csv')
mentions.head()

/var/folders/p9/9vqlyg213v3b9h9zrksgz6z00000gn/T/ipykernel_2544/2776365275.py:1: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  mentions = pd.read_csv('data/mentions.csv')


,mention_id,source,source_year,county,original_data,confidence,full_name,first_name,middle_name,last_name,...,norm_occupation,enslaver_id,location_id,head,household_id,family_id,created,narrative,soundex_last_name,narrative_vector
0,ALB-CH-1851-2071,ALB_CH_1851,1851,ALB,"{""line"": ""2071"", ""race"": ""B"", ""gender"": ""F"", ""...",0.80,Martha,Martha,NaN,NaN,...,NaN,NaN,NaN,f,NaN,NaN,2026-06-20 17:49:32.846283+00,Martha (F / B in Alb). Enslaved by: Mary Moore.,NaN,NaN
1,ALB-VR-1715-4362,ALB_VR_1715,1868,ALB,"{""line"": ""4362"", ""note"": """", ""race"": ""B"", ""typ...",0.84,Nicey Ann Coles,Nicey,Ann,Coles,...,NaN,NaN,NaN,f,NaN,NaN,2026-06-20 17:34:55.49837+00,NaN,C420,NaN
2,ALB-VR-1715-4362.1,ALB_VR_1715,1868,ALB,"{""line"": ""4362"", ""note"": """", ""race"": ""B"", ""typ...",0.85,Coles,Coles,NaN,Coles,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-20 17:36:09.428451+00,NaN,C420,NaN
3,ALB-VR-1715-4362.2,ALB_VR_1715,1868,ALB,"{""line"": ""4362"", ""note"": """", ""race"": ""B"", ""typ...",0.85,Coles,Coles,NaN,Coles,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-20 17:36:45.128086+00,NaN,C420,NaN
4,ALB-VR-1715-4362.3,ALB_VR_1715,1868,ALB,"{""line"": ""4362"", ""note"": """", ""race"": ""B"", ""typ...",0.85,Coles,Coles,NaN,Coles,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-20 17:36:51.894296+00,NaN,C420,NaN


## Look at an example of a confidence with 3

In [13]:
mentions[mentions['mention_id'] == 'ALB-CN-1870-1688' ]

,mention_id,source,source_year,county,original_data,confidence,full_name,first_name,middle_name,last_name,...,norm_occupation,enslaver_id,location_id,head,household_id,family_id,created,narrative,soundex_last_name,narrative_vector
55503,ALB-CN-1870-1688,ALB_CN_1870,1870,ALB,"{""age"": ""38"", ""head"": ""Y"", ""line"": ""1688"", ""pa...",0.9,Dabney Johnson,Dabney,NaN,Johnson,...,AGRICULTURE,NaN,NaN,t,HC1870-160,FC1870-326,2026-06-20 17:16:29.051771+00,NaN,J525,NaN


In [14]:
mentions[mentions['mention_id'] == 'ALB-CN-1880-22721' ]

,mention_id,source,source_year,county,original_data,confidence,full_name,first_name,middle_name,last_name,...,norm_occupation,enslaver_id,location_id,head,household_id,family_id,created,narrative,soundex_last_name,narrative_vector
78316,ALB-CN-1880-22721,ALB_CN_1880,1880,ALB,"{""age"": ""48"", ""head"": ""Y"", ""line"": ""22721"", ""r...",0.9,Dabney Johnson,Dabney,NaN,Johnson,...,AGRICULTURE,NaN,NaN,t,HC1880-10,FC1880-4098,2026-06-20 17:25:22.773166+00,NaN,J525,NaN


Birth year was not showing in the output so I printed them to compare

In [15]:
mentions[mentions['mention_id'] == 'ALB-CN-1870-1688' ]['birth_year']

55503    1832.0
Name: birth_year, dtype: float64

In [16]:
mentions[mentions['mention_id'] == 'ALB-CN-1880-22721' ]['birth_year']

78316    1832.0
Name: birth_year, dtype: float64

This seems to be a very good match. Birth year is exact and occupations match.

## Look at an example of confidence 2

In [17]:
truth[truth['confidence'] == 2].head(3)

,1870_line,1880_line,score,confidence,1870_id,1880_id
25,17149,25770,420,2,ALB-CN-1870-17149,ALB-CN-1880-25770
48,18766,10837,405,2,ALB-CN-1870-18766,ALB-CN-1880-10837
63,538,16327,100,2,ALB-CN-1870-538,ALB-CN-1880-16327


In [18]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1870-17149']


,mention_id,source,source_year,county,original_data,confidence,full_name,first_name,middle_name,last_name,...,norm_occupation,enslaver_id,location_id,head,household_id,family_id,created,narrative,soundex_last_name,narrative_vector
55799,ALB-CN-1870-17149,ALB_CN_1870,1870,ALB,"{""age"": ""1"", ""head"": """", ""line"": ""17149"", ""pag...",0.9,William Sammons,William,NaN,Sammons,...,NaN,NaN,NaN,f,HC1870-1294,FC1870-3316,2026-06-20 17:18:24.545488+00,NaN,S552,NaN


In [19]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1880-25770']

,mention_id,source,source_year,county,original_data,confidence,full_name,first_name,middle_name,last_name,...,norm_occupation,enslaver_id,location_id,head,household_id,family_id,created,narrative,soundex_last_name,narrative_vector
85939,ALB-CN-1880-25770,ALB_CN_1880,1880,ALB,"{""age"": ""9"", ""head"": """", ""line"": ""25770"", ""rac...",0.9,William Sammons,William,NaN,Sammons,...,DOMESTIC,NaN,NaN,f,HC1880-11,FC1880-4646,2026-06-20 17:25:46.805718+00,NaN,S552,NaN


In [20]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1870-17149']['birth_year']


55799    1869.0
Name: birth_year, dtype: float64

In [21]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1880-25770']['birth_year']

85939    1871.0
Name: birth_year, dtype: float64

Although the first and last name are the same, occupation is missing in 1870, so we cannot confirm it is the same. Birth year is 2 years apart so this is a probable match.

## Look at an Example of Confidence 1

In [22]:
truth.loc[truth['confidence'] == 1].head(3)

,1870_line,1880_line,score,confidence,1870_id,1880_id
24,17478,22724,420,1,ALB-CN-1870-17478,ALB-CN-1880-22724
52,4121,23484,100,1,ALB-CN-1870-4121,ALB-CN-1880-23484
55,12513,24928,100,1,ALB-CN-1870-12513,ALB-CN-1880-24928


In [23]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1870-17478']

,mention_id,source,source_year,county,original_data,confidence,full_name,first_name,middle_name,last_name,...,norm_occupation,enslaver_id,location_id,head,household_id,family_id,created,narrative,soundex_last_name,narrative_vector
56167,ALB-CN-1870-17478,ALB_CN_1870,1870,ALB,"{""age"": ""5"", ""head"": """", ""line"": ""17478"", ""pag...",0.9,George Johnson,George,NaN,Johnson,...,NaN,NaN,NaN,f,HC1870-1351,FC1870-3373,2026-06-20 17:18:26.306584+00,NaN,J525,NaN


In [24]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1880-22724']

,mention_id,source,source_year,county,original_data,confidence,full_name,first_name,middle_name,last_name,...,norm_occupation,enslaver_id,location_id,head,household_id,family_id,created,narrative,soundex_last_name,narrative_vector
52692,ALB-CN-1880-22724,ALB_CN_1880,1880,ALB,"{""age"": ""15"", ""head"": """", ""line"": ""22724"", ""ra...",0.9,George Johnson,George,NaN,Johnson,...,AGRICULTURE,NaN,NaN,f,HC1880-10,FC1880-4098,2026-06-20 17:25:22.773166+00,George Johnson (M born 1865 in Alb). In house ...,J525,NaN


In [25]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1870-17478']['birth_year']

56167    1865.0
Name: birth_year, dtype: float64

In [26]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1880-22724']['birth_year']

52692    1865.0
Name: birth_year, dtype: float64

## Look at Confidence at 0

In [27]:
truth.loc[truth['confidence'] == 0].head(3)

,1870_line,1880_line,score,confidence,1870_id,1880_id
53,16276,20673,100,0,ALB-CN-1870-16276,ALB-CN-1880-20673
54,10293,31262,100,0,ALB-CN-1870-10293,ALB-CN-1880-31262
58,21086,5367,100,0,ALB-CN-1870-21086,ALB-CN-1880-5367


In [28]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1870-16276']

,mention_id,source,source_year,county,original_data,confidence,full_name,first_name,middle_name,last_name,...,norm_occupation,enslaver_id,location_id,head,household_id,family_id,created,narrative,soundex_last_name,narrative_vector
81363,ALB-CN-1870-16276,ALB_CN_1870,1870,ALB,"{""age"": ""12"", ""head"": """", ""line"": ""16276"", ""pa...",0.9,James Washington,James,NaN,Washington,...,DOMESTIC,NaN,NaN,f,HC1870-1136,FC1870-3158,2026-06-20 17:18:17.542467+00,James Washington (M / B born 1858 in Alb). In ...,W252,NaN


In [29]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1870-16276']['birth_year']

81363    1858.0
Name: birth_year, dtype: float64

In [30]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1880-20673']

,mention_id,source,source_year,county,original_data,confidence,full_name,first_name,middle_name,last_name,...,norm_occupation,enslaver_id,location_id,head,household_id,family_id,created,narrative,soundex_last_name,narrative_vector
96572,ALB-CN-1880-20673,ALB_CN_1880,1880,ALB,"{""age"": ""7"", ""head"": """", ""line"": ""20673"", ""rac...",0.9,James Washington,James,NaN,Washington,...,NaN,NaN,NaN,f,HC1880-9,FC1880-3728,2026-06-20 17:25:00.183517+00,James Washington (M / B born 1873 in Alb). In ...,W252,NaN


In [31]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1880-20673']['birth_year']

96572    1873.0
Name: birth_year, dtype: float64

In this example, birth year is off by 15 years and occupation is missing in one year, so we can't compare them.

Continuing forward, I will be treating all the ground truth examples as matches, but being transparent about the confidence score.

## Comparing predicted matches to confirmed matches

Notes: I am running a old iterations of predictions that is bad so all the accuracy results will be zero. I will run it again once I have access to tge good predictions

### Threshold at 0.8

In [32]:
threshold =  0.80
threshold

0.8

In [33]:
matches = predictions[predictions['match_probability'] > threshold]
matches_pairs = list(zip(matches['unique_id_l'], matches['unique_id_r']))
matches_pairs[:3]



[('ALB-CN-1870-10570', 'ALB-CN-1880-6166'),
 ('ALB-CN-1870-12185', 'ALB-CN-1880-6171'),
 ('ALB-CN-1870-14277', 'ALB-CN-1880-6173')]

In [34]:
truth_pairs = list(zip(truth['1870_id'], truth['1880_id']))
truth_pairs[:3]

[('ALB-CN-1870-1688', 'ALB-CN-1880-22721'),
 ('ALB-CN-1870-1695', 'ALB-CN-1880-22737'),
 ('ALB-CN-1870-1693', 'ALB-CN-1880-22735')]

How much of the true pairs did we correctly match?

In [35]:
proportion_matched = len(set(matches_pairs) & set(truth_pairs)) / len(truth_pairs)
proportion_matched

0.8385269121813032

Of all the pairs we matched, how many were incorrectly matched.

#### Looking at metrics for specfic confidence levels

##### Confidence Level 3

In [36]:
truth_confidence_3 = truth[truth['confidence'] == 3]
truth_pairs_3 = list(zip(truth_confidence_3['1870_id'], truth_confidence_3['1880_id']))
truth_pairs_3[:3]


[('ALB-CN-1870-1688', 'ALB-CN-1880-22721'),
 ('ALB-CN-1870-1695', 'ALB-CN-1880-22737'),
 ('ALB-CN-1870-1693', 'ALB-CN-1880-22735')]

In [37]:
matches_pairs[:3]

[('ALB-CN-1870-10570', 'ALB-CN-1880-6166'),
 ('ALB-CN-1870-12185', 'ALB-CN-1880-6171'),
 ('ALB-CN-1870-14277', 'ALB-CN-1880-6173')]

In [38]:
proportion_matched_3 = len(set(matches_pairs) & set(truth_pairs_3)) / len(truth_pairs_3)
proportion_matched_3

0.9951456310679612

In [39]:
len(set(matches_pairs) & set(truth_pairs_3))

205

In [40]:
len(truth_pairs_3)


206

In [41]:
set(truth_pairs_3) - set(matches_pairs)

{('ALB-CN-1870-8855', 'ALB-CN-1880-17581')}

In [42]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1870-8855'].T


,67901
mention_id,ALB-CN-1870-8855
source,ALB_CN_1870
source_year,1870
county,ALB
original_data,"{""age"": ""5"", ""head"": """", ""line"": ""8855"", ""page..."
confidence,0.9
full_name,Susan A Timberlake
first_name,Susan
middle_name,A
last_name,Timberlake


In [43]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1880-17581'].T

,36986
mention_id,ALB-CN-1880-17581
source,ALB_CN_1880
source_year,1880
county,ALB
original_data,"{""age"": ""18"", ""head"": """", ""line"": ""17581"", ""ra..."
confidence,0.9
full_name,A Susan Timberlake
first_name,A
middle_name,Susan
last_name,Timberlake


This was the one match with confidence 3, we failed to match. Looking at the actual mentions record it makes sense. In the 1880 census mention, the first and middle name are flipped, so when we compare norm first name in our model we failed to catch it. This is a good point that our models will not correctly match all people due to some encoding errors.

##### Confidence Level 2

In [44]:
truth_confidence_2 = truth[truth['confidence'] == 2]
truth_pairs_2 = list(zip(truth_confidence_2['1870_id'], truth_confidence_2['1880_id']))
truth_pairs_2[:3]

[('ALB-CN-1870-17149', 'ALB-CN-1880-25770'),
 ('ALB-CN-1870-18766', 'ALB-CN-1880-10837'),
 ('ALB-CN-1870-538', 'ALB-CN-1880-16327')]

In [45]:
proportion_matched_2 = len(set(matches_pairs) & set(truth_pairs_2)) / len(truth_pairs_2)
proportion_matched_2

1.0

We successfully matched all people who are matched at confidence 2.

##### Confidence Level 1

In [46]:
truth_confidence_1 = truth[truth['confidence'] == 1]
truth_pairs_1 = list(zip(truth_confidence_1['1870_id'], truth_confidence_1['1880_id']))
truth_pairs_1[:3]

[('ALB-CN-1870-17478', 'ALB-CN-1880-22724'),
 ('ALB-CN-1870-4121', 'ALB-CN-1880-23484'),
 ('ALB-CN-1870-12513', 'ALB-CN-1880-24928')]

In [47]:
proportion_matched_1 = len(set(matches_pairs) & set(truth_pairs_1)) / len(truth_pairs_1)
proportion_matched_1

0.88

##### Confidence Level 0

In [48]:
truth_confidence_0 = truth[truth['confidence'] == 0]
truth_pairs_0 = list(zip(truth_confidence_0['1870_id'], truth_confidence_0['1880_id']))
truth_pairs_0[:3]

[('ALB-CN-1870-16276', 'ALB-CN-1880-20673'),
 ('ALB-CN-1870-10293', 'ALB-CN-1880-31262'),
 ('ALB-CN-1870-21086', 'ALB-CN-1880-5367')]

In [49]:
proportion_matched_0 = len(set(matches_pairs) & set(truth_pairs_0)) / len(truth_pairs_0)
proportion_matched_0

0.4854368932038835

In [50]:
# set(truth_pairs_0) - set(matches_pairs)
# Commited out so long output would appear it GitHub

In [51]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1870-10144'].T


,99971
mention_id,ALB-CN-1870-10144
source,ALB_CN_1870
source_year,1870
county,ALB
original_data,"{""age"": ""6"", ""head"": """", ""line"": ""10144"", ""pag..."
confidence,0.9
full_name,Letitia B Bailey
first_name,Letitia
middle_name,B
last_name,Bailey


In [52]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1880-7089'].T

,2187
mention_id,ALB-CN-1880-7089
source,ALB_CN_1880
source_year,1880
county,ALB
original_data,"{""age"": ""21"", ""head"": """", ""line"": ""7089"", ""rac..."
confidence,0.9
full_name,Mary S Bailey
first_name,Mary
middle_name,S
last_name,Bailey


It make sense that we failed to catch this match. It is a lower confidnce match due to the first name not being at all similar even though birth years are not far off and the last name is the same.

In [53]:
confidence_levels = [0, 1, 2, 3,'overall']
match_percentages = [proportion_matched_0, proportion_matched_1, proportion_matched_2, proportion_matched_3, proportion_matched]
threshold_8_df = pd.DataFrame({'confidence_level': confidence_levels, 'match_percentage': match_percentages})
threshold_8_df

,confidence_level,match_percentage
0,0,0.485437
1,1,0.880000
2,2,1.000000
3,3,0.995146
4,overall,0.838527


In [54]:
fig = go.Figure(data=[go.Table(
    header=dict(values=list(threshold_8_df.columns),
                fill_color='paleturquoise',
                align='left'),
    cells=dict(values=[threshold_8_df.confidence_level, round(threshold_8_df.match_percentage, 3)],
               fill_color='lavender',
               align='left'))
])
fig.update_layout(title= "Percent Ground Truth Matched at Match Threshold 0.8")
fig.show()


For the most part, our higher confidence corresponds for higher match percentage after setting  match threshold of 0.8.

### Threshold at 0.90

In [55]:
threshold =  0.90
threshold

0.9

In [56]:
matches = predictions[predictions['match_probability'] > threshold]
matches_pairs = list(zip(matches['unique_id_l'], matches['unique_id_r']))
matches_pairs[:3]

[('ALB-CN-1870-10570', 'ALB-CN-1880-6166'),
 ('ALB-CN-1870-12185', 'ALB-CN-1880-6171'),
 ('ALB-CN-1870-14277', 'ALB-CN-1880-6173')]

In [57]:
proportion_matched = len(set(matches_pairs) & set(truth_pairs)) / len(truth_pairs)
proportion_matched

0.8130311614730878

In [58]:
truth_confidence_3 = truth[truth['confidence'] == 3]
truth_pairs_3 = list(zip(truth_confidence_3['1870_id'], truth_confidence_3['1880_id']))
truth_pairs_3[:3]

[('ALB-CN-1870-1688', 'ALB-CN-1880-22721'),
 ('ALB-CN-1870-1695', 'ALB-CN-1880-22737'),
 ('ALB-CN-1870-1693', 'ALB-CN-1880-22735')]

In [59]:
proportion_matched_3 = len(set(matches_pairs) & set(truth_pairs_3)) / len(truth_pairs_3)
proportion_matched_3

0.9902912621359223

In [60]:
truth_confidence_2 = truth[truth['confidence'] == 2]
truth_pairs_2 = list(zip(truth_confidence_2['1870_id'], truth_confidence_2['1880_id']))
truth_pairs_2[:3]

[('ALB-CN-1870-17149', 'ALB-CN-1880-25770'),
 ('ALB-CN-1870-18766', 'ALB-CN-1880-10837'),
 ('ALB-CN-1870-538', 'ALB-CN-1880-16327')]

In [61]:
proportion_matched_2 = len(set(matches_pairs) & set(truth_pairs_2)) / len(truth_pairs_2)
proportion_matched_2

1.0

In [62]:
truth_confidence_1 = truth[truth['confidence'] == 1]
truth_pairs_1 = list(zip(truth_confidence_1['1870_id'], truth_confidence_1['1880_id']))
truth_pairs_1[:3]

[('ALB-CN-1870-17478', 'ALB-CN-1880-22724'),
 ('ALB-CN-1870-4121', 'ALB-CN-1880-23484'),
 ('ALB-CN-1870-12513', 'ALB-CN-1880-24928')]

In [63]:
proportion_matched_1 = len(set(matches_pairs) & set(truth_pairs_1)) / len(truth_pairs_1)
proportion_matched_1

0.84

In [64]:
truth_confidence_0 = truth[truth['confidence'] == 0]
truth_pairs_0 = list(zip(truth_confidence_0['1870_id'], truth_confidence_0['1880_id']))
truth_pairs_0[:3]

[('ALB-CN-1870-16276', 'ALB-CN-1880-20673'),
 ('ALB-CN-1870-10293', 'ALB-CN-1880-31262'),
 ('ALB-CN-1870-21086', 'ALB-CN-1880-5367')]

In [65]:
proportion_matched_0 = len(set(matches_pairs) & set(truth_pairs_0)) / len(truth_pairs_0)
proportion_matched_0

0.4174757281553398

In [66]:
confidence_levels = [0, 1, 2, 3,'overall']
match_percentages = [proportion_matched_0, proportion_matched_1, proportion_matched_2, proportion_matched_3, proportion_matched]
threshold_9_df = pd.DataFrame({'confidence_level': confidence_levels, 'match_percentage': match_percentages})
threshold_9_df

,confidence_level,match_percentage
0,0,0.417476
1,1,0.840000
2,2,1.000000
3,3,0.990291
4,overall,0.813031


In [67]:
fig = go.Figure(data=[go.Table(
    header=dict(values=list(threshold_8_df.columns),
                fill_color='paleturquoise',
                align='left'),
    cells=dict(values=[threshold_8_df.confidence_level, round(threshold_8_df.match_percentage, 3)],
               fill_color='lavender',
               align='left'))
])
fig.update_layout(title= "Percent Ground Truth Matched at Match Threshold 0.9")
fig.show()

For the most part, our higher confidence corresponds for higher match percentage after setting  match threshold of 0.9.